# 03 - Bronze Katmanından Silver Katmanına Geçiş

Bu notebook Bronze katmanına yazılan ham rating verisinin kontrol edilmesini, ardından film bilgileriyle zenginleştirilerek Silver katmanına taşınmasını açıklar.

## Bronze ve Silver farkı

| Katman | Açıklama |
|---|---|
| Bronze | Kafka'dan gelen ham rating verisi |
| Silver | Temizlenmiş, duplicate kayıtları azaltılmış, film adı ve türleriyle zenginleştirilmiş veri |

## Bronze kontrol script'i

In [ ]:
from pyspark.sql import SparkSession


BRONZE_PATH = "/app/delta/bronze/ratings"

spark = (
    SparkSession.builder
    .appName("CheckBronzeDelta")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

df = spark.read.format("delta").load(BRONZE_PATH)

print("Bronze tablosundaki toplam kayıt sayısı:")
print(df.count())

print("Şema:")
df.printSchema()

print("İlk 20 kayıt:")
df.show(20, truncate=False)

print("Rating dağılımı:")
df.groupBy("rating").count().orderBy("rating").show()

## Bronze kontrol kodunun işleyişi

1. Bronze Delta tablosu okunur.
2. Toplam kayıt sayısı hesaplanır.
3. Tablo şeması yazdırılır.
4. İlk 20 kayıt gösterilir.
5. Rating dağılımı hesaplanır.

Bu kontrol, streaming verinin gerçekten Delta Lake'e yazıldığını doğrulamak için kullanılır.

## Bronze → Silver dönüşüm script'i

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_unixtime, split, trim


BRONZE_PATH = "/app/delta/bronze/ratings"
SILVER_PATH = "/app/delta/silver/ratings_enriched"
MOVIES_PATH = "/app/data/raw/ml-25m/movies.csv"


spark = (
    SparkSession.builder
    .appName("BronzeToSilverMovieRatings")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


print("Bronze rating verisi okunuyor...")
ratings_df = spark.read.format("delta").load(BRONZE_PATH)

print("Movies CSV okunuyor...")
movies_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(MOVIES_PATH)
)

print("Veri temizleme ve zenginleştirme işlemleri başlıyor...")

ratings_clean_df = (
    ratings_df
    .dropna(subset=["userId", "movieId", "rating", "timestamp"])
    .dropDuplicates(["userId", "movieId", "timestamp"])
    .withColumn("rating_datetime", from_unixtime(col("timestamp")).cast("timestamp"))
)

movies_clean_df = (
    movies_df
    .dropna(subset=["movieId", "title", "genres"])
    .dropDuplicates(["movieId"])
    .withColumn("genres_array", split(col("genres"), "\\|"))
)

silver_df = (
    ratings_clean_df
    .join(movies_clean_df, on="movieId", how="inner")
    .select(
        col("userId"),
        col("movieId"),
        col("title"),
        col("genres"),
        col("genres_array"),
        col("rating"),
        col("timestamp"),
        col("rating_datetime")
    )
)

print("Silver veri örneği:")
silver_df.show(20, truncate=False)

print("Silver toplam kayıt sayısı:")
print(silver_df.count())

print("Silver Delta katmanına yazılıyor...")

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .save(SILVER_PATH)
)

print("Silver katmanı başarıyla oluşturuldu.")
print(f"Silver path: {SILVER_PATH}")

## Bronze to Silver kodunun işleyişi

1. Bronze rating tablosu okunur.
2. `movies.csv` dosyası okunur.
3. Null kayıtlar temizlenir.
4. Duplicate kayıtlar düşürülür.
5. Timestamp alanı gerçek zaman tipine çevrilir.
6. `genres` alanı `genres_array` olarak parçalanır.
7. Rating verisi ile film bilgisi `movieId` üzerinden join edilir.
8. Sonuç `/app/delta/silver/ratings_enriched` path'ine Delta formatında yazılır.

## Çalıştırma komutları

```bash
docker exec -it movie-spark bash

spark-submit --packages io.delta:delta-spark_2.12:3.2.0 check_bronze.py

spark-submit --packages io.delta:delta-spark_2.12:3.2.0 bronze_to_silver.py
```

## Silver çıktısı

Silver tablosunda şu alanlar bulunur:

- userId
- movieId
- title
- genres
- genres_array
- rating
- timestamp
- rating_datetime

Bu veri artık EDA ve modelleme için daha anlamlıdır.